# SC4001 Project E: Relation Lie Detector

The entry point for the whole pipeline. Each stage is a library in `src/` with a `run()` function; this notebook
calls them in order. Every script can also be run on its own (`python src/data/clean_data.py --help`).

| Stage | Library | Writes |
|---|---|---|
| 1. Clean | `data/clean_data.py` | `data/outputs/cleaned/clean_data.json` |
| 2. Enrich | `data/generate_left_right.py` | `data/outputs/enriched/enriched_data.json` |
| 3. Questions | `data/generate_queries.py` | `data/outputs/queries/queries_{split}.json` |
| 4. VLM run | `model/vlm.py` | `data/outputs/answers/` |
| 5. Linear probe | `training/linear_probe.py` | `model/models/linear_probe.joblib`, `training/results/linear_probe.json` |
| 6. MLP | `training/mlp.py` | `model/models/mlp.pt`, `training/results/mlp.json` |
| 7. Tune | `validation/validate.py` | `validation/results/settings.json` |
| 8. Test | `validation/test.py` | `validation/results/test.json` |
| 9. Plots | this notebook | `validation/results/figures/*.png` |

## Colab

On Colab, clones the repo and moves into `src/` so the cells below find the code. Does nothing when the notebook is
run from a local checkout. For a private repo, put a token in the URL: `https://<token>@github.com/...`.

In [ ]:
import os
from pathlib import Path

if Path("/content").exists() and not Path("/content/4001Project").exists():
    !git clone -b jeremy_new https://github.com/JeremyQuek/4001Project.git /content/4001Project
if Path("/content/4001Project/src").exists():
    os.chdir("/content/4001Project/src")
print(Path.cwd())

## Install

Installs the dependencies listed in `pyproject.toml`, so the notebook also works on Colab where the repo's
virtual environment doesn't exist. Packages that are already installed are skipped.

In [ ]:
from pathlib import Path

# the repo root holds pyproject.toml; the notebook may be started from src/ or from the root
REPO = Path.cwd().parent if (Path.cwd() / "main.ipynb").exists() else Path.cwd()
%pip install -q -e "{REPO}"

## Setup

In [ ]:
import sys
from pathlib import Path

# works whether the notebook is started from src/ or from the repo root
SRC = Path.cwd() if (Path.cwd() / "lib" / "paths.py").exists() else Path.cwd() / "src"
sys.path.insert(0, str(SRC))

from lib import paths

# On Colab, keep the raw data and the outputs on Google Drive so they survive a disconnect:
# from google.colab import drive
# drive.mount("/content/drive")
# paths.configure(vg_dir="/content/drive/MyDrive/sc4001/visual_genome",
#                 output_dir="/content/drive/MyDrive/sc4001/outputs")

print("raw data:", paths.vg_dir())
print("outputs: ", paths.output_dir())

## Download

Fetches anything missing: the Visual Genome files (about 1 GB, into `paths.vg_dir()`) and the Qwen weights (about
7.5 GB, into `model/models/`). Files that are already there are skipped. Photos are not downloaded here; stage 4
fetches each one from the URL in `image_data.json` the first time a question needs it.

In [ ]:
import urllib.request

from model import vlm

VG_URL = "https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/"
VG_FILES = ["relationships.json.zip", "objects.json.zip", "image_data.json.zip",
            "relationship_alias.txt", "object_alias.txt"]

paths.vg_dir().mkdir(parents=True, exist_ok=True)
for name in VG_FILES:
    target = paths.vg_dir() / name
    if not target.exists():
        print(f"downloading {name}")
        urllib.request.urlretrieve(VG_URL + name, target)

if not (paths.QWEN_DIR / "config.json").exists():
    vlm.download()
print("Visual Genome:", sorted(f.name for f in paths.vg_dir().iterdir()))
print("Qwen:", paths.QWEN_DIR)

## 1. Clean

Keeps relations whose predicate maps to one of the 8 labels, then filters them. `skip` switches stages off
(flag names in `clean_data.SKIP_FLAGS`). Same-name and repeated-name relations are kept, since the objects are
pointed at with drawn boxes. About 30 seconds.

In [ ]:
from data import clean_data

clean_data.run(skip=["keep_same_name", "keep_repeated_names"])

## 2. Enrich

Adds LEFT/RIGHT rows built from object box positions; Visual Genome has only about 1,300 annotated of each.
About 30 seconds.

In [ ]:
from data import generate_left_right

generate_left_right.run(pairs_per_image=2, skip=["keep_same_name", "keep_repeated_names"])

## 3. Questions

Four yes/no questions per relation (true, opposite, swapped, swapped + opposite), balanced across labels.
About 10 seconds.

In [ ]:
from data import generate_queries

generate_queries.run(train=100_000, val=20_000, test=20_000)

## 4. VLM run

Loads Qwen2.5-VL-3B-Instruct from `src/model/models/` (download once with `python src/model/vlm.py --download`).
For every question it draws the subject's box in red and the object's in blue, asks Qwen, and saves the answer,
`p_yes` / `p_no` and the hidden states at the last prompt token to `paths.answers_dir()`. Photos are downloaded
into `visual_genome/images/` as needed, using the URLs in `image_data.json`.

Work is saved in shards of 512 questions, so after a disconnect rerunning this cell carries on where it stopped.
Run a pilot first (`limit=200`) to check the answers and the speed, then run with `limit=None`.

In [ ]:
from model import vlm

vlm.run(splits=("train", "val", "test"), limit=200)  # pilot; limit=None for the full run

## 5. Linear probe

In [ ]:
from training import linear_probe

linear_probe_results = linear_probe.run()

## 6. MLP

In [ ]:
from training import mlp

mlp_results = mlp.run(seeds=(0, 1, 2))

## Compare the detectors

Test AUROC of each detector against the answer-confidence baseline.


In [ ]:
print(f"{'detector':<15}{'test AUROC':>12}{'confidence baseline':>22}")
print(f"{'linear probe':<15}{linear_probe_results['test']['auroc']:>12.4f}"
      f"{linear_probe_results['test_confidence_baseline_auroc']:>22.4f}")
print(f"{'MLP':<15}{mlp_results['test_auroc_mean']:>12.4f}{mlp_results['test_confidence_baseline_auroc']:>22.4f}")

## 7. Tune the detector + method combinations (validation split)

Four ways of using a detector at answer time, one script each in `src/validation/`:

| | Feedback loop (re-ask when flagged) | Steering (edit the hidden state in the pass) |
|---|---|---|
| Linear probe | `linear_feedback.py` | `linear_steering.py` |
| MLP | `mlp_feedback.py` | `mlp_steering.py` |

Feedback picks the P(wrong) threshold for re-asking; steering picks the strength alpha. Uses every validation question
unless `limit` is set. Qwen runs again here (one re-ask pass, plus one steered pass per alpha and detector on the
flagged questions); every pass is cached in `data/outputs/validation/`, so rerunning after a disconnect resumes.

In [ ]:
from model.vlm import VLM
from validation import validate

qwen = VLM()
settings = validate.run(limit=None, alphas=(0.5, 1.0, 2.0), vlm=qwen)

## 8. Final comparison (test split)

Baseline Qwen vs the four methods, with controls: random flags for feedback, perfect flags (upper bound), and a random
direction of the same size for steering. Settings come from stage 7; the test split is used once.

In [ ]:
from validation import test

final = test.run(limit=None, vlm=qwen)

## 9. Plots

Reads `validation/results/test.json` and saves two figures to `validation/results/figures/` for the report:

- `overview.png`: one panel per metric, each method next to its control. Blue = method, gray = control,
  orange = perfect flags (upper bound); the baseline is the zero line, or the dashed line for confidence.
- `accuracy_by_label.png`: accuracy for each relation, baseline and each method.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

from data.schemas import Predicate, TestResults

results = TestResults.model_validate_json((paths.VALIDATION_RESULTS_DIR / "test.json").read_text()).results
fig_dir = paths.VALIDATION_RESULTS_DIR / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

INK, SECONDARY, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
COLOUR = {"method": "#2a78d6", "control": MUTED, "upper": "#eb6834"}
plt.rcParams.update({"font.family": "sans-serif", "font.size": 9, "text.color": INK, "axes.labelcolor": SECONDARY,
                     "xtick.color": MUTED, "ytick.color": SECONDARY, "axes.edgecolor": AXIS,
                     "figure.facecolor": "white", "axes.facecolor": "white"})

# (row in test.json, label, kind); each method is followed by its control
ROWS = [
    ("linear_probe_feedback", "Linear probe + feedback", "method"),
    ("linear_probe_feedback_random_flags", "random flags (control)", "control"),
    ("mlp_feedback", "MLP + feedback", "method"),
    ("mlp_feedback_random_flags", "random flags (control)", "control"),
    ("perfect_flags_feedback", "Perfect flags + feedback", "upper"),
    ("linear_probe_steering", "Linear probe + steering", "method"),
    ("linear_probe_steering_random_direction", "random direction (control)", "control"),
    ("mlp_steering", "MLP + steering", "method"),
    ("mlp_steering_random_direction", "random direction (control)", "control"),
]
base = results["baseline"]
# (title, value of a row, baseline value): every panel starts its bars at 0
PANELS = [
    ("Accuracy change vs baseline (pp)", lambda m: 100 * (m.accuracy - m.baseline_accuracy), 0.0),
    ("Fixed: wrong → right", lambda m: m.fixed, 0),
    ("Broken: right → wrong", lambda m: m.broken, 0),
    ("Answers changed (%)", lambda m: 100 * m.changed_rate, 0.0),
    ("Questions acted on (%)", lambda m: 100 * (m.flagged_rate or 0), None),
    ("Mean confidence", lambda m: m.mean_confidence, base.mean_confidence),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7.5), sharey=True)
y = np.arange(len(ROWS))[::-1]
for ax, (title, value, base_value) in zip(axes.flat, PANELS):
    vals = np.array([value(results[key]) for key, _, _ in ROWS], dtype=float)
    ax.barh(y, vals, height=0.62, color=[COLOUR[kind] for _, _, kind in ROWS])
    ax.axvline(0, color=AXIS, linewidth=1)
    if base_value:
        ax.axvline(base_value, color=SECONDARY, linewidth=1.2, linestyle="--")
        ax.text(base_value, len(ROWS) - 0.35, " baseline", color=SECONDARY, fontsize=8, va="bottom")
    span = max(np.abs(vals).max(), abs(base_value or 0), 1e-9)
    for yi, v in zip(y, vals):
        # labels sit past the baseline line too, so they never collide with it
        x = max(v, base_value or v) if v >= 0 else v
        ax.text(x + (0.02 * span if v >= 0 else -0.02 * span), yi, f"{v:,.0f}" if abs(v) >= 100 else f"{v:.3g}",
                va="center",
                ha="left" if v >= 0 else "right", fontsize=8, color=SECONDARY)
    ax.set_xlim(min(0, vals.min()) - 0.15 * span, max(vals.max(), base_value or 0) + 0.18 * span)
    ax.set_title(title, loc="left", fontsize=10, color=INK)
    ax.grid(axis="x", color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.tick_params(axis="y", length=0)
for ax in axes[:, 0]:
    ax.set_yticks(y, [label for _, label, _ in ROWS])

handles = [plt.Rectangle((0, 0), 1, 1, color=COLOUR[k]) for k in ("method", "control", "upper")]
handles.append(plt.Line2D([], [], color=SECONDARY, linestyle="--"))
fig.legend(handles, ["Method", "Control", "Upper bound (perfect flags)", "Baseline"], loc="lower center",
           ncol=4, frameon=False)
fig.suptitle(f"Baseline Qwen vs the four methods on the test split (baseline accuracy {base.accuracy:.3f}, "
             f"n = {base.n:,})", x=0.01, ha="left", fontsize=12)
fig.tight_layout(rect=(0, 0.04, 1, 0.97))
fig.savefig(fig_dir / "overview.png", dpi=200)
plt.show()

# accuracy per relation: a single-hue sequential scale, every cell labelled with its value
HEAT_ROWS = [("baseline", "Baseline")] + [(key, label) for key, label, kind in ROWS if kind != "control"]
labels = list(Predicate)
grid = np.array([[results[key].accuracy_by_label.get(p, np.nan) for p in labels] for key, _ in HEAT_ROWS])
ramp = LinearSegmentedColormap.from_list("blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf",
                                                  "#184f95", "#0d366b"])
fig, ax = plt.subplots(figsize=(10, 3.8))
im = ax.imshow(grid, cmap=ramp, vmin=np.nanmin(grid), vmax=np.nanmax(grid), aspect="auto")
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        dark = im.norm(grid[i, j]) > 0.35
        ax.text(j, i, f"{grid[i, j]:.3f}", ha="center", va="center", fontsize=8, color="white" if dark else INK)
ax.set_xticks(range(len(labels)), [p.value for p in labels])
ax.set_yticks(range(len(HEAT_ROWS)), [label for _, label in HEAT_ROWS])
ax.tick_params(length=0)
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("Accuracy per relation on the test split", loc="left", fontsize=11)
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02).outline.set_visible(False)
fig.tight_layout()
fig.savefig(fig_dir / "accuracy_by_label.png", dpi=200)
plt.show()
print(f"saved figures to {fig_dir}")